<a href="https://colab.research.google.com/github/MaxineMwangi/flyrank/blob/main/w04_baseline_score_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxineMwangi/flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Work the sections **in order**. Simple words, honest numbers. Every `TODO` is something only you can fill in after looking at the real columns.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` + `flyrank/flyrank-data`.

## 0. Setup

*Load the starter CSV from the repo (no warehouse streaming needed for this one).*

In [ ]:
import os, json, subprocess
import numpy as np, pandas as pd

# Colab opens only this notebook, so pull the repo if the data folder isn't there
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank"):
        subprocess.run(["git", "clone", "-q", "https://github.com/MaxineMwangi/flyrank"], check=True)
    os.chdir("flyrank")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)
df.head()

In [ ]:
# Read the schema before touching anything
pd.DataFrame({"dtype": df.dtypes.astype(str), "n_null": df.isna().sum(), "n_unique": df.nunique()})

## Config (fill after reading the schema above)

*Everything below reads from here. Rule inputs must be **past-window only** — no future-window columns and nothing derived from the outcome.*

In [ ]:
# ---- TODO: fill these in ----
ID_COLS      = ["client_hash_id", "content_hash_id"]   # confirm these exist in the schema above
OUTCOME_COL  = "TODO_outcome_column"   # what 'went wrong / improved' later; used ONLY to CHECK signals, never as a rule input
SIGNAL_A     = "TODO_signal_a"         # past-window column behind a real FlyRank flag (staleness / CTR-vs-position / volume)
SIGNAL_B     = "TODO_signal_b"         # second past-window signal
A_HIGHER_IS_WORSE = True               # True if bigger SIGNAL_A means more urgent
B_HIGHER_IS_WORSE = True
FLAG_LINKED  = "TODO e.g. staleness behind the refresh flags"   # say which flag signal A is behind
REASON_CODE  = "TODO_reason_code"      # ONE reason code, e.g. STALE_HIGH_VALUE
ACTION_LABEL = "TODO action label"     # e.g. Refresh content
# -----------------------------
for c in ID_COLS + [OUTCOME_COL, SIGNAL_A, SIGNAL_B]:
    assert c in df.columns, f"column not found: {c}"

## 1. Signal checks + rule reasoning

*Check two signals first. One bucket table each, with n printed, then one word: CONFIRMED, OPPOSITE, MIXED or FALSE. At least one must be behind a real FlyRank flag. A clearly-explained negative is a win.*

### Signal A

In [ ]:
def bucket_table(frame, signal, outcome, q=5):
    s = frame[[signal, outcome]].dropna().copy()
    s["bucket"] = pd.qcut(s[signal], q=q, duplicates="drop")
    t = s.groupby("bucket", observed=True).agg(
        n=(outcome, "size"),
        signal_mean=(signal, "mean"),
        outcome_mean=(outcome, "mean"),
    ).round(4)
    print(f"{signal} vs {outcome}  (total n = {len(s)})")
    return t

bucket_table(df, SIGNAL_A, OUTCOME_COL)

**Signal A verdict:** TODO — one word (CONFIRMED / OPPOSITE / MIXED / FALSE)

**Why, in plain words:** TODO — what the table shows. Which flag is this signal behind: see `FLAG_LINKED`.

### Signal B

In [ ]:
bucket_table(df, SIGNAL_B, OUTCOME_COL)

**Signal B verdict:** TODO — one word

**Why, in plain words:** TODO

### My rule in one paragraph
TODO — "Rank items higher when ___ and ___, because ___. This ignores ___." Use careful words: observed, measured, directional, decision-support.

## 2. The ranked queue

*One rule: a score, ONE reason code, an action label. Written to `work/outputs/baseline_action_score.csv` from this notebook.*

In [ ]:
def pct(series, higher_is_worse):
    r = series.rank(pct=True, na_option="keep")
    return r if higher_is_worse else 1 - r

work = df.copy()
work["pct_a"] = pct(work[SIGNAL_A], A_HIGHER_IS_WORSE)
work["pct_b"] = pct(work[SIGNAL_B], B_HIGHER_IS_WORSE)

# Simple, honest rule: average of the two percentile ranks (0-1). Change the weights only if you can justify it.
W_A = 0.5
work["score"] = (W_A * work["pct_a"] + (1 - W_A) * work["pct_b"]).round(4)
work = work.dropna(subset=["score"])

work["reason_code"]  = REASON_CODE
work["action_label"] = ACTION_LABEL

queue = (work[ID_COLS + [SIGNAL_A, SIGNAL_B, "score", "reason_code", "action_label"]]
         .sort_values("score", ascending=False)
         .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

# receipts worth committing (the CSV itself stays out of git by design)
with open("work/outputs/baseline_score_metrics.json", "w") as f:
    json.dump({"n_scored": int(len(queue)), "signal_a": SIGNAL_A, "signal_b": SIGNAL_B,
               "weight_a": W_A, "reason_code": REASON_CODE, "action_label": ACTION_LABEL}, f, indent=2)

print("rows ranked:", len(queue))
queue.head(10)

## 3. Top-10 review

*For each of the top ten, one line each: the action, why it's there, and what would make it wrong.*

In [ ]:
top10 = queue.head(10)
top10[["rank"] + ID_COLS + [SIGNAL_A, SIGNAL_B, "score", "action_label"]]

| Rank | Action | Why it's there | What would make it wrong |
|---|---|---|---|
| 1 | TODO | TODO | TODO |
| 2 | TODO | TODO | TODO |
| 3 | TODO | TODO | TODO |
| 4 | TODO | TODO | TODO |
| 5 | TODO | TODO | TODO |
| 6 | TODO | TODO | TODO |
| 7 | TODO | TODO | TODO |
| 8 | TODO | TODO | TODO |
| 9 | TODO | TODO | TODO |
| 10 | TODO | TODO | TODO |

## 4. Weak picks

*Which of your top ten would you trust least, and what would you check first? Ranks 11–20 are below for comparison.*

In [ ]:
queue.iloc[10:20][["rank"] + ID_COLS + [SIGNAL_A, SIGNAL_B, "score"]]

**Weakest picks:** TODO — which ranks, and the first thing you'd check (e.g. tiny n, a missing value filled by rank, a signal that only looks high because of one outlier).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Two signal verdicts with visible bucket tables and n (at least one flag-linked)
- [ ] One rule: a score, a reason code, an action label
- [ ] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [ ] Ten reviewed rows, each with "what would make it wrong"
- [ ] No future-window or label-derived inputs in the rule
- [ ] No `TODO` text left anywhere
- [ ] Runs top to bottom with no errors (Runtime → Run all)
- [ ] Committed under `work/notebooks/` (+ metrics JSON; the CSV stays out of git) — then submit your repo URL on the card. Done.